# Baseline и поведенческие признаки

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer

ROOT = Path.cwd().parent / 'bot_detection_challenge'
sys.path.append(str(ROOT))
from metric import precision_at_recall

SEED = 42
DATE_COLS = ['cookie_created_at', 'window_start_ts', 'window_end_ts']

train = pd.read_csv(ROOT / 'data/train.csv', parse_dates=DATE_COLS)
test = pd.read_csv(ROOT / 'data/test.csv', parse_dates=DATE_COLS)
events_raw = pd.read_csv(ROOT / 'data/events.csv.gz', parse_dates=['event_ts'])
meta = pd.concat([train, test]).set_index('cookie_id')

## Очистка и валидация

Те же шаги, что в EDA.

In [2]:
start = events_raw.cookie_id.map(meta.window_start_ts)
end = events_raw.cookie_id.map(meta.window_end_ts)
events = events_raw[(events_raw.event_ts >= start) & (events_raw.event_ts < end)]

events = events.drop_duplicates().drop(columns='eid')

PLATFORM_MAP = {'android': 'android', 'ios': 'ios', 'iphone': 'ios', 'web': 'web', 'desktop': 'web'}
events['platform'] = events.platform.str.lower().map(PLATFORM_MAP)

events = events.sort_values(['cookie_id', 'event_ts'], kind='mergesort').reset_index(drop=True)
events.shape

(283833, 13)

In [3]:
days = np.sort(train.window_start_ts.unique())
folds = []
for val_days in [days[6:8], days[8:10], days[10:12], days[12:14]]:
    folds.append((np.flatnonzero(train.window_start_ts < val_days[0]),
                  np.flatnonzero(train.window_start_ts.isin(val_days))))

testlike_fold = [(np.flatnonzero(train.window_start_ts < '2026-04-13'),
                  np.flatnonzero(train.window_start_ts >= '2026-04-13'))]

y = train.target.to_numpy()


def cross_validate(model, X, y, folds, return_oof=False):
    oof = np.full(len(y), np.nan)
    for tr, va in folds:
        model.fit(X.iloc[tr], y[tr])
        oof[va] = model.predict_proba(X.iloc[va])[:, 1]

    mask = ~np.isnan(oof)
    y_val, p_val = y[mask], oof[mask]

    rng = np.random.default_rng(SEED)
    boot = [precision_at_recall(y_val[i], p_val[i]) for i in rng.integers(0, len(y_val), (1000, len(y_val)))]

    result = {
        'p_at_r70': round(precision_at_recall(y_val, p_val), 3),
        'ci95': np.percentile(boot, [2.5, 97.5]).round(3),
        'pr_auc': round(average_precision_score(y_val, p_val), 3),
        'roc_auc': round(roc_auc_score(y_val, p_val), 3),
    }
    return (result, oof) if return_oof else result

## Модели и вспомогательные функции

Модель обучается на таблице «одна строка — одна кука», а в `events` на одну куку приходится много строк. Поэтому каждый признак — это агрегат по событиям куки внутри её окна: количество, доля, среднее, разброс. Признаки одной куки не зависят от других кук, поэтому их можно считать сразу для train и test без утечки.

Модели:
- LogReg — покажет, сколько сигнала можно извлечь без нелинейностей;
- CatBoost — основная модель, т.к. сама находит нелинейные зависимости и взаимодействия признаков, а также работает с категориальными признаками и пропусками.

Связь признака с таргетом оцениваем через `univariate_auc`: ROC-AUC, если использовать признак сам по себе как скор. В отличие от корреляции Спирмена, AUC не сжимается из-за дисбаланса классов. Также будем смотреть на `coverage`: долю кук, у которых признак определён.

In [4]:
features = {}  # группа признаков -> DataFrame с индексом cookie_id


def assemble(groups):
    X = pd.concat([features[g] for g in groups], axis=1).reindex(train.cookie_id).reset_index(drop=True)
    cat_cols = X.select_dtypes('object').columns
    X[cat_cols] = X[cat_cols].fillna('missing')
    return X


def univariate_auc(F):
    """AUC каждого признака по отдельности. Меньше 0.5 — у ботов значения меньше."""
    F = F.reindex(train.cookie_id).reset_index(drop=True)
    rows = []
    for col in F.select_dtypes('number'):
        ok = F[col].notna().to_numpy()
        rows.append({'feature': col, 'auc': roc_auc_score(y[ok], F.loc[ok, col]), 'coverage': ok.mean()})
    result = pd.DataFrame(rows).set_index('feature')
    return result.sort_values('auc', key=lambda s: -abs(s - 0.5)).round(3)


def make_logreg(X):
    cat = X.select_dtypes('object').columns.tolist()
    num = [c for c in X.columns if c not in cat]
    preprocess = ColumnTransformer([
        ('num', make_pipeline(SimpleImputer(strategy='median', add_indicator=True),
                              QuantileTransformer(n_quantiles=200, output_distribution='normal', random_state=SEED)), num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
    ])
    return make_pipeline(preprocess, LogisticRegression(C=0.1, max_iter=5000))


def make_catboost(X):
    return CatBoostClassifier(iterations=800, learning_rate=0.03, depth=6, random_seed=SEED, verbose=0,
                              thread_count=-1, allow_writing_files=False,
                              cat_features=X.select_dtypes('object').columns.tolist())

## Baseline на базовых агрегатах

Базовые агрегаты — самые простые признаки, не требующие гипотез о поведении: сколько и каких событий у куки.

- `n_events`, `n_items` — число событий и уникальных объявлений;
- `cnt_<событие>` — сколько раз кука сделала событие этого типа, например `cnt_item_view` — число просмотров карточек Счётчики растут вместе с активностью;
- `share_<событие>` — доля событий конкретного типа среди всех событий куки. Доли показывают, из чего состоит поведение;
- `platform` — платформа (web, android, ios);
- `cookie_age_days` — возраст куки на начало окна, в днях.

In [5]:
def base_features(events):
    counts = pd.crosstab(events.cookie_id, events.event_name)
    shares = counts.div(counts.sum(axis=1), axis=0)
    g = events.groupby('cookie_id')
    f = pd.DataFrame({
        'n_events': g.size(),
        'n_items': g.item_id.nunique(),
        'platform': g.platform.first(),
    })
    f['cookie_age_days'] = (meta.window_start_ts - meta.cookie_created_at).dt.total_seconds() / 86400
    return f.join(counts.add_prefix('cnt_')).join(shares.add_prefix('share_'))


features['base'] = base_features(events)
univariate_auc(features['base']).head(8)

,auc,coverage
feature,,
cnt_item_view,0.685,1.0
n_events,0.665,1.0
n_items,0.665,1.0
cnt_search_results_view,0.664,1.0
cookie_age_days,0.364,1.0
cnt_seller_page_view,0.635,1.0
share_item_view,0.619,1.0
share_photo_swipe,0.383,1.0


In [6]:
X_base = assemble(['base'])
results = {
    'LogReg base': cross_validate(make_logreg(X_base), X_base, y, folds),
    'CatBoost base': cross_validate(make_catboost(X_base), X_base, y, folds),
}
pd.DataFrame(results).T

,p_at_r70,ci95,pr_auc,roc_auc
LogReg base,0.169,"[0.145, 0.213]",0.399,0.79
CatBoost base,0.231,"[0.202, 0.29]",0.501,0.815


Базовые агрегаты поднимают метрику с 0.09 (quickstart) до 0.23. Ни один признак сам по себе не разделяет классы.

## 1. Тайминги

Гипотеза: бот работает ровным циклом с короткими паузами, у человека паузы длиннее и разнообразнее.

Считаем паузы `dt` между соседними событиями куки. Паузу больше 30 минут считаем разрывом сессии; статистики `dts_*` считаются только по паузам внутри сессий.
- `dt_median`, `dts_median` — типичная пауза;
- `log_dts_std` — разброс логарифма пауз внутри сессии, то есть насколько неровный ритм. Логарифм нужен, чтобы паузы 2 и 4 с различались так же, как 200 и 400 с;
- `dts_nunique_ratio` — доля уникальных значений пауз: у ровного цикла паузы повторяются;
- `events_per_session`, `events_per_active_hour` — интенсивность;
- `share_night` — доля событий с 0 до 6 часов.

In [7]:
SESSION_GAP = 30 * 60


def timing_features(events):
    dt = events.groupby('cookie_id').event_ts.diff().dt.total_seconds()
    ev = events.assign(dt=dt, hour=events.event_ts.dt.hour)
    ev['dt_session'] = ev.dt.where(ev.dt < SESSION_GAP)
    ev['log_dt_session'] = np.log1p(ev.dt_session)
    g = ev.groupby('cookie_id')
    known = ev[ev.dt.notna()]

    f = pd.DataFrame({
        'span_hours': (g.event_ts.max() - g.event_ts.min()).dt.total_seconds() / 3600,
        'dt_min': g.dt.min(),
        'dt_q10': g.dt.quantile(0.1),
        'dt_median': g.dt.median(),
        'dts_median': g.dt_session.median(),
        'dts_mean': g.dt_session.mean(),
        'dts_std': g.dt_session.std(),
        'log_dts_std': g.log_dt_session.std(),
        'dts_nunique_ratio': g.dt_session.nunique() / g.dt_session.count(),
        'share_dt_0': (known.dt == 0).groupby(known.cookie_id).mean(),
        'share_dt_le10': (known.dt <= 10).groupby(known.cookie_id).mean(),
        'n_sessions': 1 + (ev.dt >= SESSION_GAP).groupby(ev.cookie_id).sum(),
        'n_active_hours': g.hour.nunique(),
        'share_night': (ev.hour < 6).groupby(ev.cookie_id).mean(),
    })
    f['dts_cv'] = f.dts_std / f.dts_mean
    f['events_per_session'] = g.size() / f.n_sessions
    f['events_per_active_hour'] = g.size() / f.n_active_hours
    return f


features['timing'] = timing_features(events)
univariate_auc(features['timing'])

,auc,coverage
feature,,
log_dts_std,0.246,0.921
dt_median,0.273,0.977
dts_nunique_ratio,0.285,0.969
events_per_session,0.711,1.000
dts_median,0.290,0.969
events_per_active_hour,0.694,1.000
dts_mean,0.308,0.969
dts_std,0.329,0.921
dts_cv,0.334,0.921


Гипотеза подтвердилось. Лучший признак — `log_dts_std` (разброс логарифма пауз внутри сессии): у ботов ритм ровнее. Паузы у ботов короче, а событий за сессию больше.

## 2. Поиск

Гипотеза: бот выгружает выдачу целиком — листает глубоко и повторяет один запрос.

Признаки считаем по событиям поиска:
- `page_max`, `page_mean`, `share_page_ge5` — глубина листания;
- `searches_per_query`, `share_same_query` — насколько часто повторяется один запрос;
- `share_next_page` — доля переходов на следующую страницу того же запроса;
- `n_queries`, `query_len_mean` — число разных запросов и их средняя длина.

In [8]:
def search_features(events):
    s = events[events.event_name == 'search_results_view'].copy()
    g = s.groupby('cookie_id')
    s['same_query'] = s.search_query == g.search_query.shift()
    s['next_page'] = s.same_query & (s.search_page == g.search_page.shift() + 1)
    g = s.groupby('cookie_id')

    f = pd.DataFrame({
        'n_queries': g.search_query.nunique(),
        'page_max': g.search_page.max(),
        'page_mean': g.search_page.mean(),
        'share_page_gt1': (s.search_page > 1).groupby(s.cookie_id).mean(),
        'share_page_ge5': (s.search_page >= 5).groupby(s.cookie_id).mean(),
        'share_same_query': g.same_query.mean(),
        'share_next_page': g.next_page.mean(),
        'query_len_mean': s.search_query.str.len().groupby(s.cookie_id).mean(),
    })
    f['searches_per_query'] = g.size() / f.n_queries
    return f


features['search'] = search_features(events)
univariate_auc(features['search'])

,auc,coverage
feature,,
page_mean,0.714,0.917
page_max,0.708,0.917
searches_per_query,0.706,0.917
share_page_ge5,0.680,0.917
share_same_query,0.646,0.917
share_page_gt1,0.635,0.917
n_queries,0.595,0.917
share_next_page,0.547,0.917
query_len_mean,0.517,0.917


Глубина пагинации и повторы запроса работают. Строго последовательного листания страниц почти нет.

## 3. Разнообразие

Гипотеза: сервисы мониторинга обходят много регионов и категорий, а человек ищет что-то конкретное у себя в городе.

- `n_categories`, `n_locations` — число разных категорий и городов;
- `category_entropy`, `location_entropy` — энтропия: 0, если всё в одной категории, и тем больше, чем равномернее интерес распределён;
- `top_category_share` — доля самой частой категории;
- `item_events_per_item` — сколько событий приходится на одно объявление;
- `share_pro_seller` — доля объявлений от профессиональных продавцов.

In [9]:
def entropy(events, col):
    counts = events.dropna(subset=[col]).groupby(['cookie_id', col]).size()
    p = counts / counts.groupby(level=0).transform('sum')
    return (-p * np.log(p)).groupby(level=0).sum()


def diversity_features(events):
    g = events.groupby('cookie_id')
    items = events[events.item_id.notna()]
    gi = items.groupby('cookie_id')
    category_counts = events.groupby(['cookie_id', 'item_category']).size()
    is_pro = items.seller_type.eq('pro').astype(float).where(items.seller_type.notna())

    return pd.DataFrame({
        'n_categories': g.item_category.nunique(),
        'n_locations': g.item_location.nunique(),
        'category_entropy': entropy(events, 'item_category'),
        'location_entropy': entropy(events, 'item_location'),
        'top_category_share': category_counts.groupby(level=0).max() / category_counts.groupby(level=0).sum(),
        'item_events_per_item': gi.size() / gi.item_id.nunique(),
        'share_pro_seller': is_pro.groupby(items.cookie_id).mean(),
    })


features['diversity'] = diversity_features(events)
univariate_auc(features['diversity'])

,auc,coverage
feature,,
location_entropy,0.714,0.999
n_locations,0.711,1.000
category_entropy,0.382,0.997
top_category_share,0.612,0.997
n_categories,0.394,1.000
share_pro_seller,0.558,0.978
item_events_per_item,0.512,0.984


У ботов больше локаций, но меньше категорий. Похоже на мониторинг цен: одна категория по многим регионам.

## 4. Последовательности

Гипотеза: у бота однотипный цикл «выдача → карточка → выдача», у человека больше побочных действий.

Проверить хронологию на уровне объявления («фото без просмотра карточки») не получилось: `item_id` соседних событий не связаны даже у людей. Поэтому смотрим на переходы между типами событий:
- `tr_A_to_B` — доля переходов из события A в событие B среди всех переходов куки (`srch` — выдача, `item` — карточка);
- `share_repeat_event` — доля случаев, когда следующее событие того же типа.

In [10]:
SHORT_NAMES = {'search_results_view': 'srch', 'item_view': 'item', 'photo_swipe': 'photo', 'seller_page_view': 'seller'}
TRANSITIONS = ['srch>item', 'item>srch', 'item>item', 'srch>srch', 'item>photo', 'item>seller']


def sequence_features(events):
    name = events.event_name.map(SHORT_NAMES).fillna(events.event_name)
    prev = name.groupby(events.cookie_id).shift()
    pair = prev + '>' + name
    n_transitions = pair.notna().groupby(events.cookie_id).sum()

    f = pd.crosstab(events.cookie_id, pair).reindex(columns=TRANSITIONS, fill_value=0).div(n_transitions, axis=0)
    f.columns = ['tr_' + c.replace('>', '_to_') for c in f.columns]
    f['share_repeat_event'] = (prev == name).groupby(events.cookie_id).sum() / n_transitions
    return f


features['sequence'] = sequence_features(events)
univariate_auc(features['sequence'])

,auc,coverage
feature,,
tr_item_to_item,0.625,0.977
share_repeat_event,0.624,0.977
tr_item_to_srch,0.619,0.977
tr_srch_to_item,0.610,0.977
tr_item_to_seller,0.594,0.977
tr_srch_to_srch,0.593,0.977
tr_item_to_photo,0.477,0.977


Сигнал умеренный. Боты чаще повторяют одно действие подряд.

## 5. Курсор

Гипотеза: бот либо не передаёт курсор, либо двигает его примитивно — в узкой области и мелкими шагами. Курсор есть только на web.

- `pointer_share` — доля web-событий с курсором;
- `px_*`, `py_*` — среднее, разброс и максимум координат по x и y;
- `pointer_step_mean` — среднее расстояние между соседними положениями курсора.

In [11]:
def pointer_features(events):
    web = events[events.platform == 'web']
    p = web.dropna(subset=['pointer_x'])
    step = np.hypot(p.groupby('cookie_id').pointer_x.diff(), p.groupby('cookie_id').pointer_y.diff())
    g = p.assign(step=step).groupby('cookie_id')

    return pd.DataFrame({
        'pointer_share': web.pointer_x.notna().groupby(web.cookie_id).mean(),
        'px_mean': g.pointer_x.mean(),
        'px_std': g.pointer_x.std(),
        'px_max': g.pointer_x.max(),
        'py_mean': g.pointer_y.mean(),
        'py_std': g.pointer_y.std(),
        'pointer_step_mean': g.step.mean(),
    })


features['pointer'] = pointer_features(events)
univariate_auc(features['pointer'])

,auc,coverage
feature,,
px_std,0.064,0.354
pointer_step_mean,0.068,0.354
px_max,0.108,0.365
py_std,0.194,0.354
px_mean,0.234,0.365
pointer_share,0.345,0.533
py_mean,0.449,0.365


Самые сильные признаки в задаче: `px_std` и `pointer_step_mean` с AUC ~0.93 (с обратным знаком). Но они есть только у ~35% кук, причём только в web.

## 6. User-Agent (UA)

Гипотеза: у части ботов в UA остаются следы автоматизации — `HeadlessChrome`, HTTP-библиотеки, смена UA внутри куки.

Строку целиком не используем (148 уникальных значений), а разбираем на части. 
- `ua_client` — тип клиента: браузер, приложение Авито, `HeadlessChrome` или HTTP-библиотека (`script`);
- `ua_os`, `ua_version` — ОС и версия браузера или приложения;
- `n_user_agents`, `main_ua_share` — сколько разных UA у куки и доля основного;
- `has_script_ua` — встречался ли у куки UA HTTP-библиотеки.

In [12]:
SCRIPT_UA = re.compile(r'python-requests|curl|node-fetch|urllib|Scrapy|Go-http-client', re.I)
BROWSERS = ['HeadlessChrome', 'YaBrowser', 'Edg', 'OPR', 'Firefox', 'Chrome', 'Safari']
OS_NAMES = {'Android': 'android', 'iPhone': 'ios', 'Windows': 'windows', 'Mac OS': 'mac', 'Linux': 'linux'}


def ua_client(ua):
    if ua.startswith('Avito/'):
        return 'avito_app'
    if SCRIPT_UA.search(ua):
        return 'script'
    return next((b for b in BROWSERS if b in ua), 'other')


def ua_os(ua):
    return next((name for marker, name in OS_NAMES.items() if marker in ua), 'other')


def ua_features(events):
    g = events.groupby('cookie_id').user_agent
    main_ua = g.agg(lambda s: s.value_counts().index[0])
    return pd.DataFrame({
        'ua_client': main_ua.map(ua_client),
        'ua_os': main_ua.map(ua_os),
        'ua_version': main_ua.str.extract(r'(?:Avito|Chrome|Firefox|Version)/(\d+)')[0].astype(float),
        'n_user_agents': g.nunique(),
        'main_ua_share': g.agg(lambda s: s.value_counts(normalize=True).iloc[0]),
        'has_script_ua': g.agg(lambda s: s.str.contains(SCRIPT_UA).any()).astype(int),
    })


features['ua'] = ua_features(events)
univariate_auc(features['ua'])

,auc,coverage
feature,,
ua_version,0.476,0.982
has_script_ua,0.517,1.000
n_user_agents,0.492,1.000
main_ua_share,0.507,1.000


In [13]:
client = features['ua'].ua_client.reindex(train.cookie_id).to_numpy()
train.groupby(client).target.agg(['size', 'mean']).round(3).sort_values('mean', ascending=False)

,size,mean
HeadlessChrome,286,0.262
script,200,0.220
avito_app,2141,0.092
Firefox,1428,0.083
YaBrowser,1221,0.081
Chrome,5623,0.063
Safari,192,0.047


Числовые признаки UA сигнала не несут. У `HeadlessChrome` и HTTP-библиотек доля ботов втрое выше средней, но таких кук мало, и большинство из них — всё равно люди.

## Абляция

Cumulative — добавляем группы гипотез по одной. Leave-one-out — убираем одну группу из полного набора.

In [14]:
GROUPS = ['base', 'timing', 'search', 'diversity', 'sequence', 'pointer', 'ua']


def score(groups):
    X = assemble(groups)
    return cross_validate(make_catboost(X), X, y, folds)['p_at_r70']


full = score(GROUPS)
ablation = pd.DataFrame({
    'cumulative': [score(GROUPS[:i + 1]) for i in range(len(GROUPS))],
    'without_group': [score([g for g in GROUPS if g != group]) for group in GROUPS],
}, index=GROUPS)
ablation['contribution'] = full - ablation.without_group
ablation

,cumulative,without_group,contribution
base,0.231,0.751,0.015
timing,0.407,0.575,0.191
search,0.491,0.709,0.057
diversity,0.609,0.702,0.064
sequence,0.595,0.747,0.019
pointer,0.747,0.618,0.148
ua,0.766,0.747,0.019


Главные группы — тайминги и курсор: без каждой из них метрика падает на 0.15–0.19. Разнообразие и поиск дают ещё по ~0.06.

Вклад `base`, `sequence` и `ua` около 0.02, это в пределах шума (бутстрап-интервал ±0.05). Это кандидаты на удаление при отборе признаков.

## Итог

In [15]:
X_all = assemble(GROUPS)
results['LogReg all'] = cross_validate(make_logreg(X_all), X_all, y, folds)
results['CatBoost all'], oof = cross_validate(make_catboost(X_all), X_all, y, folds, return_oof=True)
results['CatBoost all, горизонт как в test'] = cross_validate(make_catboost(X_all), X_all, y, testlike_fold)
pd.DataFrame(results).T

,p_at_r70,ci95,pr_auc,roc_auc
LogReg base,0.169,"[0.145, 0.213]",0.399,0.79
CatBoost base,0.231,"[0.202, 0.29]",0.501,0.815
LogReg all,0.583,"[0.498, 0.676]",0.703,0.916
CatBoost all,0.766,"[0.703, 0.809]",0.783,0.936
"CatBoost all, горизонт как в test",0.714,"[0.628, 0.801]",0.778,0.939


CatBoost на всех признаках даёт 0.77 против 0.09 у quickstart. LogReg на тех же признаках даёт 0.58: значит зависимости нелинейные.

На горизонте как в test метрика ниже (0.71), но PR-AUC почти такой же, а модель обучалась на вдвое меньшем объёме данных. Сильного дрейфа не видно.

## Ошибки по платформам

In [16]:
mask = ~np.isnan(oof)
res = pd.DataFrame({'y': y[mask], 'score': oof[mask], 'platform': X_all.platform[mask].to_numpy()})
threshold = res.score[res.y == 1].quantile(0.3)  # порог, при котором recall = 70%
res['flagged'] = res.score >= threshold

res.groupby('platform').apply(lambda d: pd.Series({
    'bots': d.y.sum(),
    'recall': d.flagged[d.y == 1].mean(),
    'precision': d.y[d.flagged].mean(),
    'roc_auc': roc_auc_score(d.y, d.score),
}), include_groups=False).round(3)

,bots,recall,precision,roc_auc
platform,,,,
android,150.0,0.567,0.697,0.887
ios,11.0,0.364,0.667,0.770
web,314.0,0.774,0.797,0.961


Web-ботов модель ловит хорошо, там работают признаки курсора. Слабое место — мобильные боты: курсора у них нет, и находим мы заметно меньше.

Дальше: признаки для мобильных ботов, отбор признаков, другие модели.